# Embedding and retrieval starter — planning and input inspection

This notebook inventories the delivered evidence and previews inputs for the next stage. **It does not generate embeddings, download models, create indexes, train a model or claim retrieval performance.** All executable cells use the standard library and leave files unchanged.

Implementation sequence:
1. Select source-separated passages; define token-aware chunking, context, qualifiers and stable chunk IDs.
2. Prepare independently reviewed tuning/test questions, grouped by duplicate/thread family.
3. Establish BM25, then MedCPT article/query encoding and FAISS retrieval.
4. Compare lexical, dense and fused retrieval; evaluate cross-encoder reranking separately.
5. Connect chunk IDs to graph records for bounded Neo4j retrieval and attributed LLM answers.

Use the [official MedCPT implementation](https://github.com/ncbi/MedCPT). Record model revisions and appropriate input/pooling conventions. Do not replace text with raw serialized graph JSON.


In [1]:
from pathlib import Path
import csv, json, hashlib
csv.field_size_limit(10_000_000)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'reports/kg_handoff_file_list.json').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the cloned repository or its notebooks folder.')
inventory = json.loads((ROOT / 'reports/kg_handoff_file_list.json').read_text())
GRAPH = ROOT / inventory['graph']
manifest = json.loads((GRAPH / 'manifest.json').read_text())
def node_rows(kind):
    with (GRAPH / 'nodes' / (kind + '.csv')).open() as handle:
        rows = list(csv.DictReader(handle))
    for row in rows:
        row['properties'] = json.loads(row.pop('properties_json'))
        row['evidence_ids'] = json.loads(row['evidence_ids'])
    return rows
print('Graph:', GRAPH.name)


Graph: full-graph-concepts-v1-08c99550a0fa0f86aca9


In [2]:
from collections import Counter
passages = node_rows('SourcePassage')
statements = [n for n in node_rows('LabelAssertion')
              if n['properties'].get('kind') == 'label_statement']
counts = Counter(n['properties']['source'] for n in passages)
print('Existing passage nodes:', dict(counts))
print('Captured DSLD statement nodes:', len(statements))
print('These counts are not final chunk/vector counts; text selection and chunking remain to be implemented.')


Existing passage nodes: {'natmedpro': 1179, 'reddit': 5071}
Captured DSLD statement nodes: 6463
These counts are not final chunk/vector counts; text selection and chunking remain to be implemented.


In [3]:
# Inspect stored text and provenance before designing embedding templates.
for source in sorted(counts):
    node = next(n for n in passages if n['properties']['source'] == source)
    p = node['properties']; fields = p['original_fields']
    print(json.dumps({'source': source, 'graph_node_id': node['id'],
                      'evidence_role': p.get('evidence_role'),
                      'title': fields.get('title'), 'section': fields.get('section'),
                      'text_preview': (fields.get('text') or '')[:180],
                      'evidence_ids': node['evidence_ids']}, indent=2))
statement = statements[0]
print(json.dumps({'source': 'dsld', 'graph_node_id': statement['id'],
                  'statement': statement['properties']['original_fields'],
                  'evidence_ids': statement['evidence_ids']}, indent=2))
print('Preservation does not imply relevance: marketing/disclaimers are not clinical evidence.')


{
  "source": "natmedpro",
  "graph_node_id": "natmed:0d296c0daa1cccb456da568c:s0:w0",
  "evidence_role": "clinical_reference",
  "title": "Ashwagandha",
  "section": "preamble",
  "text_preview": "Ashwagandha",
  "evidence_ids": [
    "graph_evidence:23631dfc2cb7aad564b1b4365ce26b8d12fda3000d43e05535f466bf70b9ebe8"
  ]
}
{
  "source": "reddit",
  "graph_node_id": "reddit:t1_19rg1ztatzi8z:s0:w0",
  "evidence_role": "user_report",
  "title": "",
  "section": "discussion",
  "text_preview": "I vaporise CBN hash and it\u2019s wonderful",
  "evidence_ids": [
    "graph_evidence:84a744fdc226bfa42d9bd3c9d590d1a8e2fd87ae81ab19d6592dc6be328ad631"
  ]
}
{
  "source": "dsld",
  "graph_node_id": "assertion:0010b049486c1426784ad71fd21f6259b3ef9afc8cfa672bb203fab52d978900",
  "statement": {
    "notes": "Do not use if inner safety seal is broken or missing.",
    "type": "Precautions re: All Other"
  },
  "evidence_ids": [
    "evidence:6df842566d7ec3cfc0e7a8eb1fafae6e424f63da4c65f87aec0888a6de7533

## Proposed artifacts

```text
data/retrieval/<corpus-version>/
  corpus_manifest.json
  natmed/chunks.jsonl
  dsld/chunks.jsonl
  reddit/chunks.jsonl

data/embeddings/<run-id>/
  manifest.json
  natmed/{vectors.npy,row_mapping.jsonl,index.faiss}
  dsld/{vectors.npy,row_mapping.jsonl,index.faiss}
  reddit/{vectors.npy,row_mapping.jsonl,index.faiss}
```

Each chunk should retain `chunk_id`, exact text/text hash, source, original passage offsets, graph node IDs, evidence IDs, and review/mention scope. Each vector row must explicitly map to its chunk. Reused text vectors must preserve all separate provenance links.

The manifest should record the graph/corpus hashes, encoder ID and revision, template/tokenization/pooling settings, dimension, dtype, normalization, similarity metric, index settings and software versions. Record checksums and row counts.

**Index roles:** NatMed is source-attributed clinical-reference content; DSLD is label information; Reddit is user discussion. A similarity score is not mapping confidence or clinical approval. All pending clinical statuses remain attached.

## Implementation tasks — intentionally not executed

- Choose a token budget and splitting method that preserve qualifications. Preview complete chunks before encoding.
- Install and pin the selected MedCPT/FAISS/BM25 environment; the handoff validator does not require these dependencies.
- Encode passages with MedCPT's article encoder and incoming questions with its query encoder. Validate dimensions, finite values, row mappings and nearest-neighbor examples.
- Use shared chunk IDs for BM25 and dense retrieval. Compare Recall@10 and MRR on fixed reviewed splits, then add Reciprocal Rank Fusion and cross-encoder reranking.
- Retrieve by question alone; do not seed graphs with gold answers or include preference labels/votes in model inputs.
- Preserve citations, scope and review status when expanding into graph evidence. Evaluate abstention and unsupported-answer errors separately from retrieval ranking.
- Add only new/changed text under a pinned encoder; create a new compatible run when the encoder or input templates change.

## Optional question-aware RGCN

This is a later supervised experiment, not part of the text-embedding implementation. It needs a typed graph projection, stable node/relation mappings, checked response preferences and appropriate held-out splits. Cache frozen text vectors, but not final GNN states across different questions. Compare against text/evidence baselines and keep source-support checks outside ranking scores.
